# 최종 프로젝트 3.0 — 흩어진 큐브 정리에서 "나눠서 지시하기"는 정말 유리한가?

이 노트북은 최종 프로젝트 2.0을 고친 판입니다. 처음 읽는 사람도 따로 자료를 찾지 않고 이해할 수 있도록, 무엇을 비교하는지, 2.0에서 무엇이 바뀌었는지, 왜 바꾸었는지, 그리고 여러분이 탐구할 질문을 차례로 설명합니다.

## 1. 우리가 답하려는 질문

탁자 가운데에 번호가 적힌 큐브 몇 개가 흩어져 있습니다. 로봇 팔은 이 큐브들을 번호에 따라 정해진 보관 구역으로 옮겨야 합니다(1~5번은 오른쪽 구역, 6~10번은 왼쪽 구역). 제한 시간은 300초입니다.

로봇에게 일을 시키는 방법을 세 가지로 나누어 비교합니다.

| 조건 | 로봇에게 일을 시키는 방법 | 비유 |
|---|---|---|
| **A** | 로봇을 움직이는 모델 하나에게 "탁자 위 번호 큐브를 모두 정해진 구역에 넣어라"라는 **전체 명령 한 문장만** 줍니다. 어떤 큐브부터 옮길지도 이 모델이 스스로 정해야 합니다. | 신입 직원에게 "창고 정리해 둬" 한 마디만 하는 것 |
| **B** | 사진을 보고 생각하는 모델(계획 모델)이 처음 사진 한 장을 보고 "3번 큐브를 집어 오른쪽 구역에 놓아라", "7번 큐브를 집어 왼쪽 구역에 놓아라" 같은 **짧은 지시 목록**을 만듭니다. 로봇을 움직이는 모델(실행 모델)은 이 지시를 하나씩 수행합니다. 중간에 다시 보지 않습니다. | 팀장이 처음에 할 일 목록을 적어 주고 자리를 비우는 것 |
| **C** | B와 똑같이 시작하지만, **여러분이 작성한 규칙**에 따라 중간에 사진을 다시 보고 계획을 고칠 수 있습니다. | 팀장이 일하는 중간중간 돌아와 상황을 보고 목록을 고쳐 주는 것 |

B와 C처럼 "생각하는 모델이 큰 일을 작은 지시로 나누고, 움직이는 모델이 작은 지시를 수행하는" 구조를 이 노트북에서는 **나눠서 지시하기**라고 부릅니다. 연구 질문은 이것입니다.

> **흩어진 큐브를 정리할 때, 나눠서 지시하는 방법(B, C)이 전체 명령 한 문장(A)보다 더 많이 정리하는가? 그렇다면 그 차이는 어떤 조건에서 커지는가?**

## 2. 두 종류의 모델

- **계획 모델(사진을 보고 생각하는 모델)**: 탁자 사진과 글로 된 과제 설명을 받아, 어떤 큐브를 어떤 순서로 어느 구역에 옮길지 글로 답합니다. 큐브를 집을 때 손가락을 어느 방향으로 벌릴지도 함께 정합니다. 로봇을 직접 움직이지는 못합니다.
- **실행 모델(로봇을 움직이는 모델)**: 카메라 영상 두 개(위에서 본 영상, 손목 카메라 영상)와 로봇 관절 상태, 그리고 한 문장의 지시를 받아 1초에 20번씩 팔과 손가락의 다음 움직임을 정합니다.

두 모델 모두 사람이 만든 시범(정답 움직임)을 보고 배운 신경망입니다. 이번 판에서 쓰는 모델 세 개는 모두 새로 배포했습니다(4절).

## 3. 2.0에서 무엇이 바뀌었고, 왜 바꾸었나

아래 내용은 2026년 11월 5일부터 13일까지 조교팀이 시뮬레이터에서 직접 측정한 결과에 근거합니다. 숫자는 모두 같은 시뮬레이터, 같은 300초 예산에서 얻었습니다.

### 3-1. 비교에 쓰는 장면을 "흩어진 장면"으로 바꾸었습니다

**바뀐 점.** 장면마다 흐트러진 큐브가 4개, 5개, 또는 6개 있고, 가운데 구역의 무작위 위치에 서로 중심 거리 10.5cm 이상 떨어져 놓여 있습니다. 나머지 큐브는 이미 제자리에 정리되어 있습니다(건드리면 안 됩니다). 어떤 번호가 로봇에 가장 가까운지는 장면마다 무작위입니다. 검증용 장면 24개를 `scenes_v3/scattered_validation` 폴더에 넣어 두었습니다.

**바꾼 이유.** 조교팀은 같은 큐브 다섯 개를 세 가지 모양으로 놓은 장면 36개로 먼저 실험했습니다.

- 넓게 떨어진 배치: 가운데 큐브 하나와 그 주변 큐브 네 개가 서로 4~5cm 이상 떨어져 있음
- 한쪽 방향으로 붙은 배치: 가운데 큐브의 왼쪽·오른쪽에 다른 큐브가 1cm 이내로 붙어 있음
- 사방으로 붙은 배치: 가운데 큐브의 앞·뒤·왼쪽·오른쪽 모두에 다른 큐브가 1cm 이내로 붙어 있음

그 결과(목표 큐브 60개 중 300초 뒤에도 정리되지 않고 남은 수, 적을수록 좋음)는 다음과 같았습니다.

| 배치 | A | B | C |
|---|---|---|---|
| 넓게 떨어짐 | 44 | 34 | **26** |
| 한쪽 방향으로 붙음 | 47 | 43 | 44 |
| 사방으로 붙음 | **37** | 40 | 42 |

넓게 떨어진 배치에서는 나눠서 지시하는 쪽이 분명히 나았습니다. 같은 장면끼리 짝지어 비교하면 장면 하나당 A가 C보다 평균 1.5개 더 남겼습니다(95% 신뢰구간 0.58~2.42개). 반대로 사방으로 붙은 배치에서는 A가 가장 좋았습니다. 원인을 조사해 보니 두 가지였습니다.

1. 붙은 배치는 모양이 늘 같아서 정리 순서도 늘 같았습니다. A의 모델은 이 순서를 통째로 외울 수 있었고, 큐브가 한곳에 몰려 있어 어디를 집어도 무언가가 집혔습니다.
2. 계획 모델은 큐브 사이의 1cm 이내 틈을 계획에 반영하지 못했습니다. 사방으로 붙은 장면 12개에서 처음 세운 계획을 하나씩 따라가 보면, 손가락이 들어갈 수 없는 방향으로 집으라는 지시가 포함되지 않은 계획은 0개였습니다. 흥미롭게도 계획 모델 내부의 정보를 직접 꺼내 보면 "막혀 있는지"를 90% 정도 구별할 수 있었습니다. 정보는 모델 안에 들어가 있지만, 계획을 세울 때 쓰도록 학습되지 않았던 것입니다. 이것은 아직 연구 중입니다(7절 심화 질문).

그래서 3.0의 기본 비교는 순서를 외울 수 없는 흩어진 장면으로 옮겼습니다. 붙은 장면은 심화 질문으로 남겨 둡니다.

### 3-2. A 조건에 A 전용 실행 모델을 줍니다

**바뀐 점.** 2.0의 예비 실험에서는 A, B, C가 모두 같은 실행 모델을 썼습니다. 이 모델은 "3번 큐브를 집어 오른쪽 구역에 놓아라" 같은 짧은 지시만 배웠기 때문에, 전체 명령을 받은 A는 처음부터 불리했습니다. 실제로 그 조건에서 A는 96번의 시도 중 한 번도 큐브를 옮기지 못했습니다. 3.0에서는 A에게 **전체 명령 한 문장으로 배운 모델**을 따로 줍니다.

**바꾼 이유.** 배우지 않은 일을 못 하는 것은 당연하므로, 그 실패를 "나눠서 지시하기의 효과"로 해석하면 안 됩니다. A도 자기 방식으로 충분히 배운 상태에서 비교해야 공정합니다.

**주의.** 지금 배포한 A 전용 모델은 큐브 다섯 개를 묶어 놓은 장면에서 전체 명령으로 300번 시범을 보고 배웠습니다. 흩어진 장면의 시범으로 다시 학습한 A 모델을 이어서 배포할 예정입니다. 배포되면 아래 "모델 받기" 셀만 다시 실행하면 됩니다.

### 3-3. 지시문에 "손가락을 벌리는 방향"이 들어갑니다

**바뀐 점.** 이제 지시문이 "3번 큐브를 **손가락을 앞뒤로 벌려** 집어 오른쪽 구역에 놓아라"처럼 집는 방향을 포함할 수 있습니다. 방향은 두 가지입니다. 로봇 쪽에서 보아 **좌우**(손가락이 큐브의 왼쪽과 오른쪽 면을 잡음)와 **앞뒤**(손가락이 큐브의 앞면과 뒷면을 잡음)입니다.

**바꾼 이유.** 로봇 손은 두 손가락으로 큐브의 마주 보는 두 면을 잡습니다. 큐브의 왼쪽·오른쪽에 다른 큐브가 붙어 있으면 손가락이 들어갈 틈이 없습니다. 그런 장면 12개에서 원래 방향으로 집으면 12번 모두 실패했고, 손을 90도 돌려 앞뒤로 집으면 12번 모두 성공했습니다. 흩어진 장면에서는 두 방향 모두 가능하지만, 실행 모델과 계획 모델은 이 방향 정보를 함께 배웠습니다.

### 3-4. 실행 모델을 다시 학습했습니다

**바뀐 점.** B와 C가 쓰는 실행 모델은 시범 1,923개로 추가 학습했습니다. 시범에는 손가락 방향이 들어간 지시, 그리고 한 번 집기에 실패해 큐브가 살짝 밀린 상태에서 다시 집는 장면이 들어 있습니다. 그 전 단계에서는 팔이 처음 자세가 아닌 곳에서 출발하는 시범도 616개 추가했습니다.

**바꾼 이유.** 이전 모델은 첫 번째 큐브는 잘 옮겼지만, 두 번째 큐브부터는 팔이 처음 자세가 아닌 곳에서 출발해야 해서 거의 실패했습니다(처음 자세가 아닌 곳에서 출발한 시험 34번 중 3번 성공). 출발 위치를 다양하게 한 시범을 넣자 34번 중 16번으로 올랐습니다.

### 3-5. 계획 모델을 다시 학습했습니다

**바뀐 점.** 계획 모델은 일부 큐브가 이미 정리된 장면, 그리고 손가락 방향을 포함한 답을 쓰도록 추가 학습했습니다.

**바꾼 이유.** 예전 계획 모델은 이미 제자리에 있는 큐브까지 다시 옮기라고 지시하는 일이 잦았습니다(운반 지시의 약 40%). 그런 장면을 추가로 학습하자 처음 계획에 불필요한 지시가 섞이는 비율이 46%에서 9%로 줄었습니다.

### 3-6. C 규칙의 권장 출발점을 바꾸었습니다

**바뀐 점.** 2.0의 C 예시 규칙은 "운반 하나가 끝날 때마다 사진을 비교해 실패가 의심되면 곧바로 다시 계획"하는 방식이었습니다. 3.0에서는 "**계획을 끝까지 실행한 뒤에 다시 사진을 보고 계획**"하는 방식(`final_project_v2/student_strategy_c0_v2.py`)을 출발점으로 권합니다.

**바꾼 이유.** 곧바로 다시 계획하면, 계획 모델은 방금 실패한 큐브를 또 가장 먼저 지시하는 경향이 있었습니다. 그래서 같은 큐브를 연달아 세 번 시도하다가 시도 횟수 한도에 걸려 끝나는 일이 많았습니다(11번 중 6번). 계획을 끝까지 실행한 뒤 다시 보게 하자, 넓게 떨어진 장면에서 남은 목표 큐브가 34개에서 26개로 줄었습니다. 이것이 최선이라는 뜻은 아닙니다. 더 좋은 규칙을 찾는 것이 여러분의 탐구 과제입니다(7절).

### 3-7. 결과를 읽는 방법을 정했습니다

- **같은 장면끼리 짝지어 비교합니다.** 장면마다 난이도가 다르므로 "A의 평균"과 "C의 평균"을 따로 보지 말고, 같은 장면에서 A가 남긴 수에서 C가 남긴 수를 뺀 값을 장면마다 구해 평균을 냅니다.
- **신뢰구간을 함께 씁니다.** 장면을 무작위로 다시 뽑는 계산을 수천 번 반복해, 평균 차이가 어느 범위에서 흔들리는지(95% 신뢰구간)를 봅니다. 구간이 0을 포함하면 "차이가 있다"고 말하기 어렵습니다.
- **모델의 저장 시점은 결과를 보기 전에 정합니다.** 같은 학습에서 저장한 계획 모델이라도 저장 시점에 따라, 한쪽 방향으로 붙은 장면 12개 중 실행 가능한 계획을 세운 수가 11개였다가 3개로 떨어지기도 했습니다. 평가할 장면으로 가장 좋은 저장본을 고르면 그 장면에서의 결과가 실제보다 좋게 나옵니다. 고르려면 평가 장면과 겹치지 않는 별도 장면을 쓰세요.
- **시험용 장면은 마지막에 한 번만 씁니다.** 규칙과 모델을 다 정한 뒤에만 시험 장면을 실행합니다.

## 4. 준비: 코드, 실행 환경, 모델

아래 셀은 2.0과 같은 방식으로 저장소를 준비합니다. Colab에서는 GPU 런타임을 선택한 뒤 실행하세요. 처음 실행할 때는 `DOWNLOAD_MODELS = True`로 바꾸어 이번 판의 모델 세 개를 받습니다.

| 모델 | 무엇인가 | 쓰는 조건 |
|---|---|---|
| 계획 모델 | 탁자 사진을 보고 옮길 큐브, 순서, 보관 구역, 손가락 방향을 글로 답합니다. | B, C |
| 짧은 지시용 실행 모델 | "N번 큐브를 집어 ○○ 구역에 놓아라" 한 문장을 수행합니다. | B, C |
| 전체 명령용 실행 모델 | "모든 번호 큐브를 정리하라" 한 문장만 받고 처음부터 끝까지 스스로 움직입니다. | A |

배포 정보는 저장소의 `models/release_manifest_v3_scattered_20261113.json`과 GitHub 릴리스 페이지 `models-v3-scattered-20261113`에 있습니다.

In [ ]:
import os, sys, json, platform, subprocess, shutil, time, uuid
from pathlib import Path
IS_COLAB = platform.system() == 'Linux' and Path('/content').is_dir()
TEAM = 'team01'                                    # 팀 이름(결과 폴더 이름에 쓰입니다)
SOURCE_ROOT = os.environ.get('FPV3_SOURCE', '')    # 직접 올린 vla_pipeline 경로가 있으면 지정
REPO_URL = 'https://github.com/semin126/VLA2026.git'
REPO_REF = os.environ.get('FPV3_REPO_REF', 'instructor/runtime-conda-fixes')
INSTALL = IS_COLAB and os.environ.get('FPV3_SKIP_INSTALL') != '1'
USE_DRIVE = IS_COLAB and os.environ.get('FPV3_NO_DRIVE') != '1'
WORK = Path(os.environ.get('FPV3_WORK', '/content/Final_project_v3' if IS_COLAB else 'outputs/final_project_v3'))

MANIFEST = 'models/release_manifest_v3_scattered_20261113.json'   # 이번 판의 모델 목록
MODELS_DIR = 'models_v3'                                          # 2.0 모델과 섞이지 않도록 새 폴더
SCENES_ROOT = 'scenes_v3/scattered_validation'                    # 흩어진 장면 24개

RUN_CONDITIONS = ['A', 'B', 'C']   # 실행할 조건
SECONDS = 300                      # 장면 하나의 제한 시간(초)
MAX_STEPS = 6000                   # 시뮬레이터 최대 단계 수(1단계 = 0.05초)
ACTION_STEPS = 240                 # 지시 하나에 쓰는 최대 단계 수
OBSERVER_STEPS = 60                # C에서 움직이는 중간에 상황을 확인하는 간격
MAX_CALLS = 8                      # C에서 계획 모델을 부를 수 있는 최대 횟수(B는 1회)
REPEAT_SEEDS = '42'                # 실행 모델의 무작위성 고정값(같은 값이면 같은 결과)

In [ ]:
GIT = shutil.which('git') or 'git'
if SOURCE_ROOT:
    PROJECT = Path(SOURCE_ROOT).expanduser().resolve()
else:
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates += [base, base/'vla_pipeline', base/'2026/VLA2026/vla_pipeline']
    PROJECT = next((p for p in candidates if (p/'scripts/final_project_v2.py').is_file()), None)
    if PROJECT is None:
        checkout = Path('/content' if IS_COLAB else '.') / ('fpv3_checkout_'+uuid.uuid4().hex[:8])
        subprocess.run([GIT, 'clone', REPO_URL, str(checkout)], check=True)
        subprocess.run([GIT, '-C', str(checkout), 'checkout', REPO_REF], check=True)
        PROJECT = checkout/'vla_pipeline'
PROJECT = PROJECT.resolve()
if not (PROJECT/MANIFEST).is_file():
    raise RuntimeError('3.0 모델 목록이 없습니다. 최신 브랜치를 받았는지 확인하세요: '+str(PROJECT/MANIFEST))
sys.path.insert(0, str(PROJECT))
from vlapipe.final_project_v2.notebook_runtime import NotebookRun, prepare_python, source_version
if not WORK.is_absolute():
    WORK = PROJECT/WORK
BACKUP = None
if USE_DRIVE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/VLA2026/Final_project_v3')
run = NotebookRun(PROJECT, WORK, BACKUP, TEAM)
PY = prepare_python(PROJECT, WORK, install=INSTALL)
run.python = PY
B_STRATEGY = PROJECT/'final_project_v2/baseline_strategy.py'          # B: 제공 규칙(다시 보지 않음)
C_STRATEGY = PROJECT/'final_project_v2/student_strategy_c0_v2.py'     # C: 권장 출발점. 복사해서 고치세요.
print('PROJECT:', PROJECT, '\nRUN FOLDER:', run.root)
print('SOURCE VERSION:', json.dumps(source_version(PROJECT), ensure_ascii=False))
run.run('scripts/check_env.py')

In [ ]:
DOWNLOAD_MODELS = False    # 처음 한 번은 True로 바꾸어 실행하세요(약 4.6GB).
UPGRADE_MODELS = False     # 새 A 모델이 배포되었을 때 True로 바꾸면 이전 모델을 백업하고 교체합니다.
if DOWNLOAD_MODELS:
    options = ['--manifest', MANIFEST, '--models-dir', MODELS_DIR, '--roles', 'vlm,vla_hier,vla_single']
    if UPGRADE_MODELS:
        options.append('--upgrade')
    run.run('scripts/download_models.py', *options)
VLM_MODEL = str(PROJECT/MODELS_DIR/'vlm/pretrained_model')
VLA_SUBTASK = str(PROJECT/MODELS_DIR/'vla_hier/pretrained_model')
VLA_SCENE = str(PROJECT/MODELS_DIR/'vla_single/pretrained_model')
for p in (VLM_MODEL, VLA_SUBTASK, VLA_SCENE):
    print(('준비됨  ' if Path(p).is_dir() else '없음    ') + p)

## 5. 장면 살펴보기

흩어진 장면 24개를 불러오고 몇 개를 그림으로 봅니다. 각 장면 파일에는 흐트러진 큐브의 번호 목록(`roles` → `misplaced`)이 들어 있습니다. 이 큐브들이 이번 과제의 "목표 큐브"입니다.

In [ ]:
from IPython.display import display, Image, HTML
SCENES = (PROJECT/SCENES_ROOT).resolve()
registry = json.loads((SCENES/'suite.json').read_text(encoding='utf-8'))
VALID_SCENES = [SCENES/r['path'] for r in registry['scenes'] if r['split'] == 'validation']
print('장면 수:', len(VALID_SCENES))
for path in VALID_SCENES[:3]:
    spec = json.loads(path.read_text(encoding='utf-8'))
    print(path.stem, '흐트러진 큐브:', spec['roles']['misplaced'])
    display(Image(filename=str(path.with_suffix('')/'scene.png'), width=420))

## 6. 실행하기

### 6-1. 한 장면에서 A, B, C를 차례로 실행

먼저 장면 하나로 세 조건이 모두 돌아가는지 확인합니다. 결과 폴더에는 조건마다 `report.json`(최종 상태와 점수), `events.jsonl`(무슨 일이 언제 일어났는지), 그리고 B·C에는 `plan_001.json`(계획 모델에게 준 글과 받은 답)이 남습니다.

In [ ]:
SCENE = VALID_SCENES[0]
TRIALS = run.root/('trials_'+uuid.uuid4().hex[:8])
def trial(condition, scene=SCENE):
    args = ['trial', '--condition', condition, '--scene', scene, '--out', TRIALS/Path(scene).stem/condition,
            '--seconds', SECONDS, '--max-steps', MAX_STEPS, '--action-steps', ACTION_STEPS,
            '--observer-steps', OBSERVER_STEPS, '--max-calls', MAX_CALLS,
            '--checkpoint', VLA_SCENE if condition == 'A' else VLA_SUBTASK]
    if condition in 'BC':
        args += ['--strategy', B_STRATEGY if condition == 'B' else C_STRATEGY, '--vlm-model', VLM_MODEL]
    run.run('scripts/final_project_v2.py', *args)
for condition in RUN_CONDITIONS:
    trial(condition)

### 6-2. 24개 장면 전체를 실행

전체 실행은 조건 하나당 장면 24개 × 최대 300초이므로 GPU에서 약 2시간씩 걸립니다. 같은 장면, 같은 무작위성 고정값으로 A, B, C를 모두 실행해야 짝지어 비교할 수 있습니다. 실행 중에는 같은 GPU로 다른 작업을 하지 마세요. 제한 시간이 실제 시계로 재기 때문에 다른 작업이 끼어들면 결과가 나빠집니다.

In [ ]:
RUN_SUITE = False   # 준비가 끝나면 True
if RUN_SUITE:
    SUITE_TRIALS = run.root/('suite_'+uuid.uuid4().hex[:8])
    args = ['suite', '--scenes', SCENES, '--split', 'validation', '--conditions', ','.join(RUN_CONDITIONS),
            '--out', SUITE_TRIALS, '--seconds', SECONDS, '--max-steps', MAX_STEPS,
            '--action-steps', ACTION_STEPS, '--observer-steps', OBSERVER_STEPS, '--max-calls', MAX_CALLS,
            '--repeat-seeds', REPEAT_SEEDS, '--checkpoint', VLA_SUBTASK, '--a-checkpoint', VLA_SCENE,
            '--vlm-model', VLM_MODEL, '--strategy', C_STRATEGY, '--b-strategy', B_STRATEGY, '--no-video']
    run.run('scripts/final_project_v2.py', *args)
    print('결과 폴더:', SUITE_TRIALS)

### 6-3. 결과 읽기: 같은 장면끼리 짝지어 비교하기

아래 셀은 장면마다 "목표 큐브 중 300초 뒤에도 정리되지 않은 수"를 세고, 같은 장면에서 A − C, B − C를 계산합니다. 값이 양수이면 C가 더 많이 정리했다는 뜻입니다. 이미 정리되어 있던 큐브를 흐트러뜨린 경우도 따로 셉니다. 흐트러진 큐브 수(4/5/6)별로도 나누어 보여 줍니다.

In [ ]:
import random
from collections import defaultdict

def analyse(root):
    left, disturbed, k_of = {}, {}, {}
    for rep in sorted(Path(root).glob('scene_*/*/*/report.json')):
        r = json.loads(rep.read_text(encoding='utf-8'))
        spec = json.loads((rep.parent/'scene.json').read_text(encoding='utf-8'))
        targets = spec['roles']['misplaced']
        fin, ini = r['final']['objects'], r['initial']['objects']
        key = (spec['id'], r['condition'])
        left[key] = sum(not fin[n]['correct'] for n in targets)
        disturbed[key] = sum(ini[n]['correct'] and not fin[n]['correct'] for n in ini if n not in targets)
        k_of[spec['id']] = len(targets)
    scenes = sorted(k_of)
    conds = sorted({c for _, c in left})
    print('조건별 합계 (목표 큐브 중 남은 수 / 정리돼 있던 큐브를 흐트러뜨린 수)')
    for c in conds:
        print(f'  {c}: 남음 {sum(left.get((s, c), 0) for s in scenes)} / 전체 목표 {sum(k_of.values())},'
              f' 흐트러뜨림 {sum(disturbed.get((s, c), 0) for s in scenes)}')

    def paired(other, subset):
        d = [left[(s, other)] - left[(s, 'C')] for s in subset if (s, other) in left and (s, 'C') in left]
        if not d:
            return None
        rng = random.Random(0)
        means = sorted(sum(rng.choice(d) for _ in d) / len(d) for _ in range(4000))
        return {'장면 수': len(d), '평균 차이': round(sum(d) / len(d), 2),
                '95% 신뢰구간': [round(means[100], 2), round(means[3899], 2)],
                'C가 나은 장면': sum(x > 0 for x in d), '상대가 나은 장면': sum(x < 0 for x in d)}
    for other in ('A', 'B'):
        if other not in conds or 'C' not in conds:
            continue
        print(f'\n{other} − C (양수면 C가 더 많이 정리)')
        print('  전체:', paired(other, scenes))
        for k in (4, 5, 6):
            print(f'  흐트러진 큐브 {k}개:', paired(other, [s for s in scenes if k_of[s] == k]))

if RUN_SUITE:
    analyse(SUITE_TRIALS)

## 7. 여러분이 탐구할 질문

정답은 제공하지 않습니다. 질문마다 "무엇을 재면 답할 수 있는지"만 안내합니다. 결과는 반드시 같은 장면끼리 짝지어 비교하고, 신뢰구간을 함께 적으세요.

### 질문 1. 흩어진 장면에서 나눠서 지시하기는 정말 유리한가?
- 24개 장면에서 A, B, C를 모두 실행하고 6-3의 표를 만드세요.
- C가 나은 장면과 A가 나은 장면을 각각 두세 개 골라 영상을 보세요. 무엇이 달랐나요?

### 질문 2. 흐트러진 큐브가 많을수록 차이가 커지는가?
- 흐트러진 큐브가 4개, 5개, 6개인 장면을 나누어 A − C를 비교하세요.
- 가설 예시: "큐브가 많을수록 순서를 정하는 일이 어려워지므로, 순서를 계획 모델이 대신 정해 주는 B·C가 더 유리해진다." 이 가설이 맞는지 확인하세요.
- 장면 수가 작은 묶음은 신뢰구간이 넓습니다. 결론의 강도를 그에 맞게 조절하세요.

### 질문 3. C가 실패한 이유는 계획 때문인가, 실행 때문인가?
- C가 큐브를 남긴 장면마다 `plan_001.json`의 계획과 영상을 비교해 다음 중 하나로 분류하세요.
  (가) 계획이 틀렸다: 없는 번호, 이미 정리된 큐브, 잘못된 구역
  (나) 집기에 실패했다
  (다) 집었지만 엉뚱한 곳에 놓았다
  (라) 시간이 모자랐다
- 분류표를 보고, 계획 모델과 실행 모델 중 어느 쪽을 개선해야 효과가 클지 판단하세요.

### 질문 4. 더 좋은 C 규칙을 만들 수 있는가?
- `student_strategy_c0_v2.py`를 복사해 새 이름으로 저장하고 고친 뒤, 설정 셀의 `C_STRATEGY`를 바꾸세요.
- 규칙이 할 수 있는 일은 "계속 진행 / 다시 계획 / 멈춤"을 고르는 것, 그리고 다시 계획할 때 계획 모델에게 짧은 메모를 전하는 것입니다. 계획의 순서를 직접 바꿀 수는 없습니다.
- 생각해 볼 점:
  - 언제 다시 보는 것이 좋은가: 매번? 실패가 의심될 때만? 계획을 다 쓴 뒤에만?
  - 방금 실패한 큐브를 곧바로 다시 시도하지 않게 하려면 어떻게 해야 하는가?
  - 계획 모델을 한 번 부르는 데 몇 초가 드는가? 남은 시간이 그보다 짧으면 어떻게 할 것인가?
- 규칙을 바꿀 때는 한 번에 한 가지만 바꾸고 비교하세요. 그래야 무엇이 효과를 냈는지 알 수 있습니다.

### 질문 5. 300초를 어디에 쓰는가?
- `events.jsonl`과 `report.json`에서 계획 모델을 부른 횟수와 걸린 시간, 로봇이 움직인 시간을 구하세요.
- 계획에 쓴 시간이 길수록 결과가 나빠지는가? C가 B보다 계획 모델을 더 자주 부르는 대가는 무엇인가?

### 질문 6. 가까운 큐브부터 옮기는 것이 항상 좋은가?
- 계획 모델이 세운 순서가 "로봇에 가까운 순서"와 얼마나 같은지 세어 보세요.
- 가까운 큐브부터 옮기면 팔이 덜 움직이지만, 옮기는 도중 다른 큐브를 건드릴 수도 있습니다. 순서와 결과의 관계를 찾아보세요.

### 질문 7(심화). 큐브가 서로 붙어 있으면 왜 나눠서 지시하기가 이기지 못했나?
- 3-1의 표에서 사방으로 붙은 배치에서는 A가 가장 좋았습니다. 조교팀이 찾은 두 가지 원인(순서를 외울 수 있었음, 계획 모델이 좁은 틈을 계획에 쓰지 못함)을 다시 읽고, 다음을 생각해 보세요.
  - 붙은 배치의 모양을 장면마다 다르게 하면 A의 장점이 사라질까?
  - 계획 모델이 틈을 "보고는 있지만 쓰지 못한다"는 것은 어떤 뜻일까? 어떤 학습 자료를 주면 쓰게 될까?
  - 집기에 실패했을 때 그 사실을 계획 모델에게 알려 주면(질문 4의 메모), 다른 방향으로 집거나 옆 큐브를 먼저 치우는 계획을 세울 수 있을까?

### 공정한 비교를 위한 점검표
- [ ] A, B, C를 같은 장면, 같은 무작위성 고정값, 같은 제한 시간으로 실행했다.
- [ ] A에는 전체 명령으로 배운 모델을, B·C에는 짧은 지시로 배운 모델을 썼다.
- [ ] 모델과 규칙은 결과를 보기 전에 정했고, 평가 장면으로 고르지 않았다.
- [ ] 실패한 시험과 오류가 난 시험도 지우지 않고 기록했다.
- [ ] 결론에 장면 수와 신뢰구간을 함께 적었다.
- [ ] 실행 중에 같은 GPU로 다른 작업을 하지 않았다.

## 8. 지금까지 조교팀이 측정한 결과

| 실험 | 장면 | 결과(목표 큐브 중 남은 수, 적을수록 좋음) |
|---|---|---|
| 큐브 다섯 개 배치 세 가지 | 넓게 떨어짐 12개 | A 44, B 34, C 26 (목표 60개 중) |
| | 한쪽 방향으로 붙음 12개 | A 47, B 43, C 44 |
| | 사방으로 붙음 12개 | A 37, B 40, C 42 |
| 흩어진 장면 | 24개 | 측정 중. A 전용 모델을 흩어진 장면 시범으로 다시 학습한 뒤 실행합니다. 결과가 나오면 이 표를 갱신합니다. |

모든 결과는 실행 모델의 무작위성 고정값 42 한 가지로 얻었습니다. 고정값을 바꾸어 반복하면 결과가 달라질 수 있습니다. 같은 장면을 반복 실행한 것은 독립된 장면을 더 모은 것과 같지 않다는 점도 기억하세요.